# Korean Olympiad entropy inspection

This Colab notebook performs only the uncertainty-inspection stage:

1. Load the 20-cluster Olympiad dataset.
2. Deterministically sample 250 problems from every cluster (5,000 total).
3. Generate solutions with EXAONE-4.0-1.2B while measuring predictive token entropy.
4. Save overall and per-cluster histograms, quantile summaries, generation lengths, and token-limit rates.

This notebook deliberately does **not** assign D1–D5 labels or choose cutoff values. Its output is intended for deciding those cutoffs in the next step. Long-running generation is checkpointed to Google Drive.

In [ ]:
%pip install -q -U datasets huggingface_hub transformers accelerate pandas pyarrow matplotlib seaborn tqdm

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
print("Using HF_TOKEN from Colab Secrets" if HF_TOKEN else "HF_TOKEN not found; using public access")

In [ ]:
# ---- User controls ----
CLUSTERED_DATASET_PATH = (
    "/content/drive/MyDrive/Korean-TDCS/data/olympiad_clustering/"
    "numina_olympiads_clustered.parquet"
)
DRIVE_ROOT = "/content/drive/MyDrive/Korean-TDCS/data/olympiad_entropy"
SAMPLES_PER_CLUSTER = 250
RANDOM_SEED = 42

MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
GENERATION_BATCH_SIZE = 256  # Sized for a 96 GB RTX PRO 6000 Blackwell GPU.
MAX_PROMPT_TOKENS = 1536
MAX_NEW_TOKENS = 512
CHECKPOINT_ROWS = 1024
TAIL_TOKEN_COUNT = 16
SAVE_FULL_GENERATIONS = True

HISTOGRAM_METRIC = "mean_normalized_entropy"
HISTOGRAM_BINS = 50
DOWNLOAD_GENERATIONS_PARQUET = False

## 1. Validate GPU and create output paths

In [ ]:
import gc
import hashlib
import json
import math
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. Select a GPU runtime before continuing.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}")
print(f"GPU VRAM: {gpu.total_memory / 1024**3:.1f} GiB")
print(f"CUDA capability: {gpu.major}.{gpu.minor}")
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
os.environ["TOKENIZERS_PARALLELISM"] = "true"

drive_root = Path(DRIVE_ROOT)
drive_root.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(drive_root / "hf_cache")
os.environ["HF_DATASETS_CACHE"] = str(drive_root / "hf_cache" / "datasets")

CANDIDATES_PATH = drive_root / "entropy_candidates_250_per_cluster.parquet"
OVERALL_SUMMARY_PATH = drive_root / "entropy_overall_summary.csv"
CLUSTER_SUMMARY_PATH = drive_root / "entropy_per_cluster_summary.csv"
COMPACT_METRICS_PATH = drive_root / "entropy_metrics.csv"
OVERALL_HISTOGRAM_PATH = drive_root / "entropy_histogram_overall.png"
CLUSTER_HISTOGRAM_PATH = drive_root / "entropy_histograms_by_cluster.png"
LENGTH_DIAGNOSTICS_PATH = drive_root / "entropy_length_diagnostics.png"
RUN_SUMMARY_PATH = drive_root / "entropy_run_summary.json"
DOWNLOAD_BUNDLE_PATH = drive_root / "olympiad_entropy_summary_outputs.zip"

## 2. Sample exactly 250 problems per cluster

In [ ]:
from datasets import Dataset, load_dataset

clustered_path = Path(CLUSTERED_DATASET_PATH)
if not clustered_path.exists():
    raise FileNotFoundError(f"Run the clustering notebook first; missing: {clustered_path}")
clustered_dataset = load_dataset("parquet", data_files=str(clustered_path), split="train")
if "cluster_id" not in clustered_dataset.column_names:
    raise ValueError("The clustered dataset has no cluster_id column")

cluster_labels = np.asarray(clustered_dataset["cluster_id"], dtype=np.int32)
cluster_ids = sorted(np.unique(cluster_labels).tolist())
rng = np.random.default_rng(RANDOM_SEED)
selected_indices = []
for cluster_id in cluster_ids:
    available = np.flatnonzero(cluster_labels == cluster_id)
    if len(available) < SAMPLES_PER_CLUSTER:
        raise ValueError(
            f"Cluster {cluster_id} has {len(available)} rows, fewer than {SAMPLES_PER_CLUSTER}"
        )
    selected_indices.extend(rng.choice(available, size=SAMPLES_PER_CLUSTER, replace=False).tolist())

problem_texts_ko = list(clustered_dataset["problem_ko"])
# Length ordering greatly reduces left-padding waste during batched generation.
selected_indices = sorted(selected_indices, key=lambda index: len(str(problem_texts_ko[index] or "")))
candidates = clustered_dataset.select(selected_indices)
candidates = candidates.add_column("original_dataset_index", selected_indices)
candidates = candidates.add_column("candidate_id", list(range(len(candidates))))
candidates.to_parquet(str(CANDIDATES_PATH))

sample_counts = pd.Series(candidates["cluster_id"]).value_counts().sort_index()
print(f"Clusters: {len(cluster_ids)}")
print(f"Candidates: {len(candidates):,}")
print(sample_counts.to_string())
if not (sample_counts == SAMPLES_PER_CLUSTER).all():
    raise AssertionError("Sampling is not balanced across clusters")

## 3. Create a resumable generation run

The run identifier depends on the candidate problems, model, prompt, and token limits—but not GPU batch size. You can reduce or increase `GENERATION_BATCH_SIZE` without invalidating completed checkpoint blocks.

In [ ]:
SOLVING_INSTRUCTION = (
    "다음 수학 문제를 단계적으로 풀어 주세요. 필요한 계산과 논리를 설명하고, "
    "마지막 줄에는 반드시 '정답: ...' 형식으로 최종 답을 쓰세요.\n\n문제:\n"
)

def candidate_fingerprint(dataset):
    digest = hashlib.sha256()
    for candidate_id, problem in zip(dataset["candidate_id"], dataset["problem_ko"]):
        digest.update(str(candidate_id).encode("ascii"))
        digest.update(str(problem or "").encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()

generation_config = {
    "candidate_fingerprint": candidate_fingerprint(candidates),
    "candidate_rows": len(candidates),
    "model_id": MODEL_ID,
    "solving_instruction": SOLVING_INSTRUCTION,
    "max_prompt_tokens": MAX_PROMPT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "tail_token_count": TAIL_TOKEN_COUNT,
    "save_full_generations": SAVE_FULL_GENERATIONS,
}
run_id = hashlib.sha256(json.dumps(generation_config, sort_keys=True).encode()).hexdigest()[:12]
run_dir = drive_root / f"generation_{run_id}"
checkpoint_dir = run_dir / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
(run_dir / "config.json").write_text(json.dumps(generation_config, ensure_ascii=False, indent=2), encoding="utf-8")
GENERATIONS_PARQUET_PATH = run_dir / "candidate_generations_with_entropy.parquet"

checkpoint_blocks = []
for start in range(0, len(candidates), CHECKPOINT_ROWS):
    end = min(start + CHECKPOINT_ROWS, len(candidates))
    checkpoint_blocks.append((start, end, checkpoint_dir / f"part_{start:05d}_{end:05d}.parquet"))

def valid_checkpoint(path, start, end):
    if not path.exists():
        return False
    try:
        part = load_dataset("parquet", data_files=str(path), split="train")
        return len(part) == end - start and part[0]["candidate_id"] == start and part[-1]["candidate_id"] == end - 1
    except Exception:
        return False

missing_blocks = [block for block in checkpoint_blocks if not valid_checkpoint(block[2], block[0], block[1])]
print(f"Generation run: {run_id}")
print(f"Checkpoint blocks complete: {len(checkpoint_blocks) - len(missing_blocks)}/{len(checkpoint_blocks)}")

## 4. Generate solutions and measure exact predictive entropy

For each generated position, entropy is calculated from the model's full next-token probability distribution. The notebook records mean entropy, entropy normalized by vocabulary size, mean greedy-token surprisal, and mean entropy over the final 16 generated tokens.

In [ ]:
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    LogitsProcessor,
    LogitsProcessorList,
)

class TokenEntropyRecorder(LogitsProcessor):
    def __init__(self):
        self.entropies = []
        self.greedy_surprisals = []

    def __call__(self, input_ids, scores):
        logits = scores.float()
        log_partition = torch.logsumexp(logits, dim=-1)
        probabilities = torch.softmax(logits, dim=-1)
        entropy = log_partition - (probabilities * logits).sum(dim=-1)
        greedy_surprisal = log_partition - logits.max(dim=-1).values
        self.entropies.append(entropy.detach())
        self.greedy_surprisals.append(greedy_surprisal.detach())
        return scores

def format_prompt(problem):
    messages = [{"role": "user", "content": SOLVING_INSTRUCTION + str(problem)}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

if missing_blocks:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,
        device_map={"": "cuda:0"},
        attn_implementation="sdpa",
    )
    model.eval()
    vocabulary_size = int(model.config.vocab_size)
    entropy_normalizer = math.log(vocabulary_size)
    eos_value = model.generation_config.eos_token_id
    eos_ids = set(eos_value if isinstance(eos_value, list) else [eos_value])
    eos_ids.discard(None)
    print(f"Model device: {model.device}; dtype: {model.dtype}; vocabulary: {vocabulary_size:,}")
    print(f"Generation batch size: {GENERATION_BATCH_SIZE}; max new tokens: {MAX_NEW_TOKENS}")

    for block_start, block_end, block_path in tqdm(missing_blocks, desc="Checkpoint blocks"):
        block_rows = []
        try:
            for batch_start in range(block_start, block_end, GENERATION_BATCH_SIZE):
                batch_end = min(batch_start + GENERATION_BATCH_SIZE, block_end)
                batch = candidates.select(range(batch_start, batch_end))
                prompts = [format_prompt(problem) for problem in batch["problem_ko"]]
                encoded = tokenizer(
                    prompts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=MAX_PROMPT_TOKENS,
                ).to("cuda:0")
                recorder = TokenEntropyRecorder()
                with torch.inference_mode():
                    sequences = model.generate(
                        **encoded,
                        max_new_tokens=MAX_NEW_TOKENS,
                        do_sample=False,
                        use_cache=True,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=model.generation_config.eos_token_id,
                        logits_processor=LogitsProcessorList([recorder]),
                    )

                generated_ids = sequences[:, encoded["input_ids"].shape[1] :]
                entropies = torch.stack(recorder.entropies, dim=1).float().cpu().numpy()
                surprisals = torch.stack(recorder.greedy_surprisals, dim=1).float().cpu().numpy()
                generated_ids_cpu = generated_ids.cpu().numpy()

                for local_index in range(len(batch)):
                    token_ids = generated_ids_cpu[local_index]
                    generated_length = len(token_ids)
                    encountered_eos = False
                    for position, token_id in enumerate(token_ids):
                        if int(token_id) in eos_ids:
                            generated_length = position + 1
                            encountered_eos = True
                            break
                    generated_length = max(1, min(generated_length, entropies.shape[1]))
                    entropy_values = entropies[local_index, :generated_length]
                    surprisal_values = surprisals[local_index, :generated_length]
                    tail_count = min(TAIL_TOKEN_COUNT, generated_length)
                    decoded = tokenizer.decode(token_ids[:generated_length], skip_special_tokens=True)
                    source_row = batch[local_index]
                    block_rows.append({
                        "candidate_id": int(source_row["candidate_id"]),
                        "original_dataset_index": int(source_row["original_dataset_index"]),
                        "cluster_id": int(source_row["cluster_id"]),
                        "generated_tokens": int(generated_length),
                        "hit_token_limit": bool(not encountered_eos and generated_length >= MAX_NEW_TOKENS),
                        "mean_token_entropy_nats": float(entropy_values.mean()),
                        "mean_normalized_entropy": float(entropy_values.mean() / entropy_normalizer),
                        "tail_token_entropy_nats": float(entropy_values[-tail_count:].mean()),
                        "mean_greedy_surprisal_nats": float(surprisal_values.mean()),
                        "generation": decoded if SAVE_FULL_GENERATIONS else decoded[-500:],
                    })
                del encoded, sequences, generated_ids, recorder
        except torch.cuda.OutOfMemoryError as error:
            torch.cuda.empty_cache()
            raise RuntimeError(
                f"CUDA OOM at batch size {GENERATION_BATCH_SIZE}. Reduce GENERATION_BATCH_SIZE and rerun; "
                "completed checkpoint blocks will be reused."
            ) from error

        Dataset.from_list(block_rows).to_parquet(str(block_path))
        print(f"Saved checkpoint {block_path.name}: {len(block_rows):,} rows")

    del model
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("All generation checkpoint blocks already exist; model loading is skipped.")

## 5. Combine checkpoints and verify completeness

In [ ]:
generation_parts = [
    load_dataset("parquet", data_files=str(path), split="train")
    for _, _, path in checkpoint_blocks
]
generation_rows = []
for part in generation_parts:
    generation_rows.extend(part.to_list())
generation_rows.sort(key=lambda row: row["candidate_id"])

if len(generation_rows) != len(candidates):
    raise ValueError(f"Expected {len(candidates)} generation rows, found {len(generation_rows)}")
if [row["candidate_id"] for row in generation_rows] != list(range(len(candidates))):
    raise ValueError("Generation candidate IDs are incomplete or duplicated")

generation_dataset = Dataset.from_list(generation_rows)
generation_dataset.to_parquet(str(GENERATIONS_PARQUET_PATH))
metrics = generation_dataset.to_pandas()
metrics.drop(columns=["generation"]).to_csv(COMPACT_METRICS_PATH, index=False)
print(f"Complete generation rows: {len(metrics):,}")
print(f"Saved generations: {GENERATIONS_PARQUET_PATH}")

## 6. Print entropy summaries

The 20th, 40th, 60th, and 80th percentiles are included as references only. This notebook does not apply them as difficulty cutoffs.

In [ ]:
QUANTILES = [0.00, 0.01, 0.05, 0.10, 0.20, 0.25, 0.40, 0.50, 0.60, 0.75, 0.80, 0.90, 0.95, 0.99, 1.00]

def summarize_frame(frame, metric):
    values = frame[metric].astype(float)
    summary = {
        "rows": len(frame),
        "mean": values.mean(),
        "std": values.std(),
        "token_limit_rate": frame["hit_token_limit"].mean(),
        "mean_generated_tokens": frame["generated_tokens"].mean(),
    }
    for quantile, value in values.quantile(QUANTILES).items():
        summary[f"p{int(round(quantile * 100)):02d}"] = value
    return summary

overall_summary = pd.DataFrame([summarize_frame(metrics, HISTOGRAM_METRIC)])
overall_summary.insert(0, "metric", HISTOGRAM_METRIC)
cluster_summaries = []
for cluster_id, frame in metrics.groupby("cluster_id", sort=True):
    row = summarize_frame(frame, HISTOGRAM_METRIC)
    row["cluster_id"] = int(cluster_id)
    cluster_summaries.append(row)
cluster_summary = pd.DataFrame(cluster_summaries).sort_values("cluster_id")

overall_summary.to_csv(OVERALL_SUMMARY_PATH, index=False)
cluster_summary.to_csv(CLUSTER_SUMMARY_PATH, index=False)
display(overall_summary)
display(cluster_summary)

## 7. Plot overall and per-cluster entropy histograms

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
values = metrics[HISTOGRAM_METRIC].astype(float)
reference_quantiles = values.quantile([0.2, 0.4, 0.6, 0.8])

fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(values, bins=HISTOGRAM_BINS, kde=True, ax=ax)
for quantile, value in reference_quantiles.items():
    ax.axvline(value, linestyle="--", linewidth=1.5, label=f"p{int(quantile * 100)} = {value:.4f}")
ax.set_title(f"Overall distribution: {HISTOGRAM_METRIC} ({len(values):,} problems)")
ax.set_xlabel(HISTOGRAM_METRIC)
ax.legend()
fig.tight_layout()
fig.savefig(OVERALL_HISTOGRAM_PATH, dpi=180, bbox_inches="tight")
plt.show()

cluster_count = metrics["cluster_id"].nunique()
columns = 4
rows = math.ceil(cluster_count / columns)
fig, axes = plt.subplots(rows, columns, figsize=(18, 3.8 * rows), sharex=True)
axes = np.asarray(axes).reshape(-1)
common_range = (values.min(), values.max())
for axis, (cluster_id, frame) in zip(axes, metrics.groupby("cluster_id", sort=True)):
    cluster_values = frame[HISTOGRAM_METRIC].astype(float)
    axis.hist(cluster_values, bins=30, range=common_range, alpha=0.85)
    axis.axvline(cluster_values.median(), color="red", linestyle="--", linewidth=1)
    axis.set_title(f"Cluster {cluster_id} | n={len(frame)} | median={cluster_values.median():.4f}")
    axis.set_xlabel(HISTOGRAM_METRIC)
for axis in axes[cluster_count:]:
    axis.axis("off")
fig.suptitle("Entropy distributions by problem cluster (shared x-axis)", fontsize=16, y=1.002)
fig.tight_layout()
fig.savefig(CLUSTER_HISTOGRAM_PATH, dpi=180, bbox_inches="tight")
plt.show()

## 8. Check length and token-limit confounding

Entropy may be distorted if many generations hit the token limit or if it strongly correlates with generation length. These diagnostics should be considered before choosing cutoffs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].hexbin(
    metrics["generated_tokens"], metrics[HISTOGRAM_METRIC], gridsize=45, mincnt=1, cmap="viridis"
)
axes[0].set_xlabel("Generated tokens")
axes[0].set_ylabel(HISTOGRAM_METRIC)
axes[0].set_title("Entropy versus generation length")
token_limit_by_cluster = metrics.groupby("cluster_id")["hit_token_limit"].mean()
axes[1].bar(token_limit_by_cluster.index.astype(str), token_limit_by_cluster.values)
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Token-limit rate")
axes[1].set_ylim(0, 1)
axes[1].set_title(f"Rate of reaching {MAX_NEW_TOKENS} generated tokens")
fig.tight_layout()
fig.savefig(LENGTH_DIAGNOSTICS_PATH, dpi=180, bbox_inches="tight")
plt.show()

print(f"Overall token-limit rate: {metrics['hit_token_limit'].mean():.2%}")
print(f"Entropy/length Pearson correlation: {metrics[HISTOGRAM_METRIC].corr(metrics['generated_tokens']):.4f}")

## 9. Save and download the entropy summary outputs

In [ ]:
from google.colab import files

run_summary = {
    "run_id": run_id,
    "model_id": MODEL_ID,
    "clusters": len(cluster_ids),
    "samples_per_cluster": SAMPLES_PER_CLUSTER,
    "total_candidates": len(candidates),
    "generation_batch_size": GENERATION_BATCH_SIZE,
    "max_prompt_tokens": MAX_PROMPT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "histogram_metric": HISTOGRAM_METRIC,
    "overall_token_limit_rate": float(metrics["hit_token_limit"].mean()),
    "entropy_length_correlation": float(metrics[HISTOGRAM_METRIC].corr(metrics["generated_tokens"])),
    "generations_parquet": str(GENERATIONS_PARQUET_PATH),
}
RUN_SUMMARY_PATH.write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(run_summary, ensure_ascii=False, indent=2))

bundle_files = [
    RUN_SUMMARY_PATH, OVERALL_SUMMARY_PATH, CLUSTER_SUMMARY_PATH, COMPACT_METRICS_PATH,
    OVERALL_HISTOGRAM_PATH, CLUSTER_HISTOGRAM_PATH, LENGTH_DIAGNOSTICS_PATH,
]
with zipfile.ZipFile(DOWNLOAD_BUNDLE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in bundle_files:
        archive.write(path, arcname=path.name)
files.download(str(DOWNLOAD_BUNDLE_PATH))

if DOWNLOAD_GENERATIONS_PARQUET:
    files.download(str(GENERATIONS_PARQUET_PATH))
else:
    print(f"Full generations remain saved at: {GENERATIONS_PARQUET_PATH}")